In [1]:
import math
import seaborn as sns
import numpy as np
import scanpy as sc
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
from abc_atlas_access.abc_atlas_cache.abc_project_cache import AbcProjectCache

## Loading & processing data

In [2]:
# Selecting the brain region
select_region = "Isoctx-CTX-Glut"

In [3]:
# Loading AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
expr_path = base_path / f"AnnData/WMB-10Xv3-{select_region}-raw-wmeta-filtered.h5ad"
adata = sc.read_h5ad(expr_path)
adata

AnnData object with n_obs × n_vars = 44121 × 32285
    obs: 'cell_barcode', 'barcoded_cell_sample_label', 'library_label', 'feature_matrix_label', 'entity', 'brain_section_label', 'library_method', 'region_of_interest_acronym', 'donor_label', 'donor_genotype', 'donor_sex', 'dataset_label', 'x', 'y', 'cluster_alias', 'neurotransmitter', 'class', 'subclass', 'supertype', 'cluster', 'neurotransmitter_color', 'class_color', 'subclass_color', 'supertype_color', 'cluster_color', 'region_of_interest_order', 'region_of_interest_color'
    var: 'gene_symbol'

In [4]:
gene_names_df = adata.var.copy()
adata.var.set_index("gene_symbol", inplace=True)
# adata.var_names_make_unique()

In [5]:
# Preprocess the data
# adata.raw = adata  # Store the raw data
sc.pp.normalize_total(adata, target_sum=1e4)  # Normalize counts per cell
sc.pp.log1p(adata)  # Log-transform the data
sc.pp.highly_variable_genes(adata, n_top_genes=2000, subset=True)  # Select highly variable genes

### DE

In [6]:
# Perform differential expression analysis
group_by="supertype"
sc.tl.rank_genes_groups(
    adata,
    groupby=group_by,
    method="wilcoxon",
    use_raw=False
)

In [7]:
# Extract the differential expression results
deg_results = pd.DataFrame({
    group: pd.DataFrame(adata.uns["rank_genes_groups"]["names"])[group]
    for group in adata.uns["rank_genes_groups"]["names"].dtype.names
})

# Save the dataframe to a CSV file
output_path = base_path / "outputs/DEG/" / f"WMB-10Xv3-{select_region}-DEG-{group_by}.csv"
deg_results.to_csv(output_path, index=False)

print(f"Differentially expressed genes saved to {output_path}")

Differentially expressed genes saved to /data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/outputs/DEG/WMB-10Xv3-Isoctx-CTX-Glut-DEG-supertype.csv


In [8]:
deg_results.head()

,0023 L4/5 IT CTX Glut_1,0024 L4/5 IT CTX Glut_2,0025 L4/5 IT CTX Glut_3,0026 L4/5 IT CTX Glut_4,0027 L4/5 IT CTX Glut_5,0028 L4/5 IT CTX Glut_6
0,Galntl6,Cux1,Tox,Synpr,Lingo2,Tenm3
1,Slc24a3,Pld5,Pcp4,Syt17,Alcam,Tenm2
2,Pcp4,Atp1a1,Grik3,Atp2b4,Kctd1,Astn2
3,Cntn5,Calb1,Etv1,Fxyd6,Etl4,Unc5d
4,S100b,Ddit4l,Tmsb10,Camk2d,Ptprt,Frmd5


In [9]:
# Saving processed AnnData object
base_path = Path("/data/scRNA/ABCA/AIBS/AWS/expression_matrices/WMB-10Xv3/20230630/")
save_path = base_path / f"processed/WMB-10Xv3-{select_region}-raw-wmeta-filtered-DEG-{group_by}.h5ad"
sc.write(save_path, adata)